# Benemérita Universidad Autónoma de Puebla (BUAP)
## Facultad de Ciencias de la Computación | Licenciatura en Ingeniería en Ciencia de Datos
### Asignatura: Estadística para Ciencia de Datos (ICDA 250)
---
# Unidad 4: Pruebas de Hipótesis
## Guía Didáctica y Laboratorio Computacional para el Docente

---

### Enfoque Pedagógico de la Unidad:
En la **Unidad 3** aprendimos a cuantificar la incertidumbre y estimar parámetros poblacionales desconocidos mediante estimadores puntuales e intervalos de confianza. En esta **Unidad 4: Pruebas de Hipótesis**, damos el paso hacia la **toma de decisiones basada en datos**:
> *¿Son las diferencias que observamos en los datos producto de un efecto real o simplemente ruido y variabilidad muestral aleatoria?*

Esta guía está diseñada para que tú, como profesor, conduzcas tu clase de forma dinámica y aplicada. Desmitificaremos el concepto del **valor $p$**, los errores de decisión ($\alpha$ y $\beta$), la potencia estadística y los diferentes contrastes paramétricos y no paramétricos utilizando el dataset real de **Costos Médicos y Seguros de Salud (`insurance.csv`)**.

### Mapa Temático Oficial:
1. **4.1 Formulación de hipótesis**: Hipótesis nula ($H_0$), alternativa ($H_1$), nivel de significancia $\alpha$, estadísticos de prueba y valor $p$.
2. **4.2 Errores Tipo I y Tipo II**: La matriz de confusión estadística y los costos de equivocarse.
3. **4.3 Potencia estadística ($1-\beta$)**: Tamaño del efecto ($d$ de Cohen) y cálculo del tamaño muestral necesario.
4. **4.4 Tipos de pruebas estadísticas**:
   - **4.4.1 Pruebas para medias**: Una muestra ($t$-test), dos muestras independientes ($t$ de Welch) y ANOVA de una vía ($F$-test).
   - **4.4.2 Pruebas para proporciones**: Prueba $Z$ de una proporción y diferencia de proporciones en cohortes.
   - **4.4.3 Pruebas de independencia**: Prueba de Chi-cuadrada ($\chi^2$) de Pearson en tablas de contingencia.
   - **4.4.4 Pruebas de bondad de ajuste y normalidad**: Diagnóstico con Shapiro-Wilk y contraste robusto no paramétrico (Mann-Whitney $U$).
5. **4.8 Aplicaciones Prácticas y Dashboard Actuarial**: Síntesis visual y toma de decisiones en Ciencia de Datos en Salud.
6. **Orientaciones Pedagógicas y Preguntas Socráticas**: Guía para detonar el debate crítico en el aula.


## 1. Configuración del Entorno y Carga del Dataset

Cargamos las librerías analíticas fundamentales (`scipy.stats`, `statsmodels`, `pandas`, `seaborn`) y exploramos el dataset `insurance.csv` ($N = 1,338$ registros), que contiene información sobre edad, género, índice de masa corporal (`bmi`), número de hijos, hábito de fumar (`smoker`), región geográfica y costos médicos anuales (`charges`).


In [ ]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.api as sm
from statsmodels.stats.power import TTestIndPower
from statsmodels.stats.proportion import proportions_ztest
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración visual
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["font.size"] = 10
plt.rcParams["axes.titlesize"] = 12
plt.rcParams["axes.labelsize"] = 11

# Carga de datos
df = pd.read_csv("insurance.csv")

print("Ecosistema analítico configurado exitosamente.")
print(f"Dimensiones del dataset: {df.shape[0]} pólizas de seguros y {df.shape[1]} variables.")
print(f"Valores nulos en el dataset: {df.isnull().sum().sum()} (Dataset 100% limpio)")
print("-" * 65)
print("Resumen de variables numéricas:")
print(df[["age", "bmi", "children", "charges"]].describe().round(2))
df.head(4)


## 2. Tema 4.1: Formulación de Hipótesis y la Mecánica del Valor $p$

### Marco Teórico Didáctico:
Una **prueba de hipótesis** es una regla matemática de decisión entre dos afirmaciones opuestas:
1. **Hipótesis Nula ($H_0$)**: Representa el *status quo*, la ausencia de efecto o que no hay diferencia. Se asume provisionalmente verdadera hasta que la evidencia demuestre lo contrario.
2. **Hipótesis Alternativa ($H_1$ o $H_a$)**: La afirmación o reclamo que el científico de datos sospecha o desea respaldar (ej. "los fumadores pagan significativamente más").

### La Lógica del Valor $p$ (*p-value*):
> El **valor $p$** es la probabilidad de haber obtenido un estadístico de prueba tan extremo o más extremo que el observado en nuestra muestra, **asumiendo que la hipótesis nula ($H_0$) es verdadera**.

$$\text{Regla de Oro: Si } p < \alpha \implies \text{Rechazamos } H_0 \quad (\text{Efecto estadísticamente significativo})$$

Donde $\alpha$ es el **nivel de significancia** predefinido (usualmente $\alpha = 0.05$, es decir, tolerancia a un 5% de falsas alarmas).

---
### Ejercicio 1: Visualización Interactiva de las Zonas de Rechazo y el Estadístico


In [ ]:
# Visualización conceptual de una prueba de hipótesis bilateral (Two-tailed test)
alpha = 0.05
gl = 50  # Grados de libertad representativos
x = np.linspace(-4, 4, 1000)
y = stats.t.pdf(x, df=gl)

# Valores críticos para alpha = 0.05 bilateral
t_crit_izq = stats.t.ppf(alpha / 2, df=gl)
t_crit_der = stats.t.ppf(1 - alpha / 2, df=gl)

# Supongamos un estadístico muestral observado t_obs = 2.45
t_obs = 2.45
p_val = 2 * (1 - stats.t.cdf(abs(t_obs), df=gl))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(x, y, label=f"Distribución t bajo H0 (gl={gl})", color="#2c3e50", lw=2)

# Regiones de rechazo sombreadas (alpha/2 en cada cola)
ax.fill_between(x, y, where=(x <= t_crit_izq), color="#e74c3c", alpha=0.5, label=f"Zona de Rechazo (alpha/2 = {alpha/2})")
ax.fill_between(x, y, where=(x >= t_crit_der), color="#e74c3c", alpha=0.5)

# Zona de no rechazo
ax.fill_between(x, y, where=((x > t_crit_izq) & (x < t_crit_der)), color="#2ecc71", alpha=0.15, label=f"Zona de Retención H0 ({1-alpha:.0%})")

# Línea del estadístico observado
ax.axvline(t_obs, color="#8e44ad", lw=2.5, linestyle="--", label=f"Estadístico Observado: t = {t_obs:.2f}")

ax.set_title("Mecánica de Decisión en Prueba de Hipótesis Bilateral", fontsize=13, fontweight="bold")
ax.set_xlabel("Escala del Estadístico t")
ax.set_ylabel("Densidad de Probabilidad")
ax.legend(loc="upper right", frameon=True)

print("=" * 65)
print("         PARÁMETROS DE LA PRUEBA DIDÁCTICA")
print("=" * 65)
print(f"Nivel de significancia (alpha):  {alpha:.2f}")
print(f"Valores críticos t:              [{t_crit_izq:.3f}, {t_crit_der:.3f}]")
print(f"Estadístico t observado:         {t_obs:.3f}")
print(f"Valor p calculado:               {p_val:.4f} ({p_val*100:.2f}%)")
print(f"Decisión formal:                 {'RECHAZAR H0 (Significativo)' if p_val < alpha else 'NO RECHAZAR H0'}")
print("=" * 65)

plt.tight_layout()
plt.show()


## 3. Tema 4.2: Errores Tipo I ($\alpha$) y Tipo II ($\beta$)

En la inferencia estadística nunca tenemos certeza absoluta del 100%, solo probabilidad. Al tomar una decisión sobre $H_0$, existen 4 escenarios posibles organizados en una **Matriz de Confusión Estadística**:

| Realidad Poblacional \ Decisión Tomada | No Rechazar $H_0$ | Rechazar $H_0$ |
| :--- | :---: | :---: |
| **$H_0$ es Verdadera (No hay efecto)** | **Decisión Correcta** (Confianza: $1 - \alpha$) | **Error Tipo I ($\alpha$)** (Falso Positivo) |
| **$H_0$ es Falsa (Sí hay efecto real)** | **Error Tipo II ($\beta$)** (Falso Negativo) | **Decisión Correcta** (Potencia: $1 - \beta$) |

### Analogía Didáctica en Seguros Médicos:
- **Error Tipo I ($\alpha$)**: Una aseguradora concluye erróneamente que los asegurados de cierta región tienen mayor riesgo cuando en realidad su costo es idéntico al promedio nacional. *Consecuencia*: Aumento injustificado de primas y pérdida de clientes.
- **Error Tipo II ($\beta$)**: La aseguradora no detecta que los pacientes con obesidad extrema (`bmi > 35`) generan sobrecostos reales. *Consecuencia*: Cobrar una prima insuficiente y sufrir pérdidas financieras graves.

---
### Ejercicio 2: Gráfica del Solapamiento de Hipótesis y el Trade-off $\alpha \leftrightarrow \beta$


In [ ]:
# Graficamos dos distribuciones: H0 (Realidad Nula) vs H1 (Realidad Alternativa)
x = np.linspace(-4, 7, 1000)

mu_0 = 0.0     # Centro bajo H0
mu_1 = 2.5     # Centro bajo H1 (efecto real)
sigma = 1.0

y0 = stats.norm.pdf(x, mu_0, sigma)
y1 = stats.norm.pdf(x, mu_1, sigma)

# Punto de corte crítico para alpha = 0.05 unilateral
x_crit = stats.norm.ppf(0.95, loc=mu_0, scale=sigma)

# Beta es el área bajo H1 a la izquierda de x_crit
beta = stats.norm.cdf(x_crit, loc=mu_1, scale=sigma)
potencia = 1 - beta

fig, ax = plt.subplots(figsize=(9, 4.5))

ax.plot(x, y0, label="H0: Sin Efecto Real (mu=0)", color="#2980b9", lw=2)
ax.plot(x, y1, label="H1: Con Efecto Real (mu=2.5)", color="#8e44ad", lw=2)

# Error Tipo I (alpha): cola derecha de H0
ax.fill_between(x, y0, where=(x >= x_crit), color="#e74c3c", alpha=0.5, label=f"Error Tipo I (alpha = 0.05)")

# Error Tipo II (beta): cola izquierda de H1
ax.fill_between(x, y1, where=(x <= x_crit), color="#f39c12", alpha=0.4, label=f"Error Tipo II (beta = {beta:.3f})")

# Potencia (1 - beta): cola derecha de H1
ax.fill_between(x, y1, where=(x >= x_crit), color="#27ae60", alpha=0.25, label=f"Potencia (1 - beta = {potencia:.3f})")

ax.axvline(x_crit, color="black", linestyle="--", lw=1.5, label=f"Umbral Crítico c = {x_crit:.2f}")

ax.set_title("Distribución de Errores Tipo I, Tipo II y Potencia Estadística", fontsize=13, fontweight="bold")
ax.set_xlabel("Escala de Medida del Parámetro")
ax.set_ylabel("Densidad")
ax.legend(loc="upper right", frameon=True)

print("=" * 65)
print("         TRADE-OFF DE ERRORES ESTADÍSTICOS")
print("=" * 65)
print(f"Nivel de Significancia (Error Tipo I alpha):  0.050 (5.0%)")
print(f"Probabilidad de Falso Negativo (beta):       {beta:.3f} ({beta*100:.1f}%)")
print(f"Potencia Estadística (1 - beta):             {potencia:.3f} ({potencia*100:.1f}%)")
print("=" * 65)

plt.tight_layout()
plt.show()


## 4. Tema 4.3: Potencia Estadística ($1-\beta$) y Tamaño del Efecto ($d$ de Cohen)

### Concepto Clave:
La **Potencia Estadística** ($1 - \beta$) es la capacidad de nuestro experimento o análisis para detectar una diferencia cuando realmente existe.
En la práctica de Ciencia de Datos:
- Un estudio con **baja potencia** ($< 80\%$) es un desperdicio de recursos porque no podrá confirmar efectos reales.
- El estándar internacional de la industria exige una potencia mínima del **80% ($1-\beta \ge 0.80$)**.

### Tamaño del Efecto ($d$ de Cohen):
No basta con decir si algo es "significativo"; necesitamos saber **qué tan grande es el impacto práctico**:
$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{\text{pooled}}}$$
*Criterio de Jacob Cohen*:
- $d \approx 0.2$: Efecto pequeño.
- $d \approx 0.5$: Efecto mediano.
- $d \ge 0.8$: Efecto grande.

---
### Ejercicio 3: Cálculo del $d$ de Cohen en `charges` y Curva de Potencia Muestral


In [ ]:
# 1. Calculamos el tamaño del efecto real entre fumadores y no fumadores en insurance.csv
fumadores = df[df["smoker"] == "yes"]["charges"]
no_fumadores = df[df["smoker"] == "no"]["charges"]

n1, n2 = len(fumadores), len(no_fumadores)
s1, s2 = fumadores.std(ddof=1), no_fumadores.std(ddof=1)
m1, m2 = fumadores.mean(), no_fumadores.mean()

# Desviación estándar combinada (pooled standard deviation)
s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
d_cohen = (m1 - m2) / s_pooled

print("=" * 65)
print("     TAMAÑO DEL EFECTO: FUMADORES VS NO FUMADORES")
print("=" * 65)
print(f"Costo Medio Fumadores:       ${m1:,.2f} USD (n={n1})")
print(f"Costo Medio No Fumadores:    ${m2:,.2f} USD (n={n2})")
print(f"Diferencia Absoluta:         +${m1 - m2:,.2f} USD")
print(f"d de Cohen Calculado:        {d_cohen:.3f}")
print("Diagnóstico de Cohen:        ¡EFECTO MASIVO (d >> 0.8)!")
print("=" * 65)

# 2. Análisis de Curvas de Potencia Estadística según tamaño de muestra
analizador_potencia = TTestIndPower()
tamanos_muestra = np.linspace(5, 120, 100)
efectos = [0.2, 0.5, 0.8]  # Pequeño, Mediano, Grande
colores = ["#e67e22", "#3498db", "#27ae60"]

fig, ax = plt.subplots(figsize=(9, 4.5))

for efecto, color in zip(efectos, colores):
    potencias = [analizador_potencia.power(effect_size=efecto, nobs1=n, alpha=0.05, ratio=1.0) for n in tamanos_muestra]
    ax.plot(tamanos_muestra, potencias, label=f"Efecto Cohen d = {efecto}", color=color, lw=2.2)

ax.axhline(0.80, color="red", linestyle="--", lw=1.5, label="Umbral Estándar de Potencia (80%)")
ax.set_title("Curva de Potencia Estadística vs Tamaño Muestral (alpha = 0.05)", fontsize=13, fontweight="bold")
ax.set_xlabel("Tamaño de Muestra por Grupo (n)")
ax.set_ylabel("Potencia Estadística (1 - beta)")
ax.legend(loc="lower right", frameon=True)

plt.tight_layout()
plt.show()


## 5. Tema 4.4.1: Pruebas de Hipótesis para Medias

Cubrimos tres contrastes fundamentales en la práctica analítica:
1. **Prueba $t$ para una muestra**: Compara la media de una muestra frente a un estándar de referencia ($\mu_0$).
2. **Prueba $t$ de Welch para dos muestras independientes**: Compara dos grupos sin asumir varianzas iguales.
3. **Análisis de Varianza (ANOVA de una vía)**: Compara las medias de 3 o más grupos simultáneamente mediante el estadístico $F$ de Fisher.

---
### Ejercicio 4: Evaluación de Costos Médicos con Pruebas $t$ y ANOVA


In [ ]:
# -------------------------------------------------------------
# 1. Prueba t de Una Muestra: ¿Los costos promedio difieren de $12,000 USD?
# H0: mu = 12000 vs H1: mu != 12000
# -------------------------------------------------------------
mu_benchmark = 12000.0
t_stat_1s, p_val_1s = stats.ttest_1samp(df["charges"], popmean=mu_benchmark)

# -------------------------------------------------------------
# 2. Prueba t de Welch (2 Muestras): Costos Fumadores vs No Fumadores
# H0: mu_fumador = mu_nofumador vs H1: mu_fumador != mu_nofumador
# -------------------------------------------------------------
t_stat_welch, p_val_welch = stats.ttest_ind(fumadores, no_fumadores, equal_var=False)

# -------------------------------------------------------------
# 3. ANOVA de Una Vía: ¿Difieren los costos médicos según la Región?
# H0: mu_NE = mu_NW = mu_SE = mu_SW vs H1: Al menos una difiere
# -------------------------------------------------------------
regiones = [df[df["region"] == r]["charges"] for r in df["region"].unique()]
f_stat_anova, p_val_anova = stats.f_oneway(*regiones)

print("=" * 70)
print("          RESULTADOS DE PRUEBAS DE HIPÓTESIS PARA MEDIAS")
print("=" * 70)
print("1. PRUEBA t DE UNA MUESTRA (Charges vs $12,000 Benchmark):")
print(f"   Media Muestral: ${df['charges'].mean():,.2f} USD")
print(f"   Estadístico t:  {t_stat_1s:.3f} | p-valor: {p_val_1s:.4e}")
print(f"   Conclusión:     {'Rechazamos H0 (Difiere significativamente)' if p_val_1s < 0.05 else 'No se rechaza H0'}")
print("-" * 70)
print("2. PRUEBA t DE WELCH PARA DOS MUESTRAS (Fumadores vs No Fumadores):")
print(f"   Estadístico t:  {t_stat_welch:.3f} | p-valor: {p_val_welch:.4e}")
print(f"   Conclusión:     {'Rechazamos H0 (Impacto masivo de tabaquismo)' if p_val_welch < 0.05 else 'No se rechaza H0'}")
print("-" * 70)
print("3. ANOVA DE UNA VÍA (Costo a través de 4 Regiones):")
print(f"   Estadístico F:  {f_stat_anova:.3f} | p-valor: {p_val_anova:.4f}")
print(f"   Conclusión:     {'Rechazamos H0 (Hay diferencias entre regiones)' if p_val_anova < 0.05 else 'No hay diferencia significativa'}")
print("=" * 70)

# Gráfico de Boxplot por Región y Tabaquismo
fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=df, x="region", y="charges", hue="smoker", palette=["#2ecc71", "#e74c3c"], ax=ax)
ax.set_title("Distribución de Costos Médicos por Región y Condición de Tabaquismo", fontweight="bold")
ax.set_ylabel("Cargos Médicos (USD)")
ax.set_xlabel("Región Geográfica")
plt.tight_layout()
plt.show()


## 6. Tema 4.4.2: Pruebas de Hipótesis para Proporciones

En variables categóricas o eventos binarios evaluamos tasas de éxito o prevalencias mediante el **test $Z$ de proporciones**:
1. **Una proporción**: Evaluamos si la tasa de fumadores en nuestra cartera difiere de la referencia del **15% ($p_0 = 0.15$)**.
2. **Dos proporciones**: Comparamos la tasa de fumadores entre **hombres vs mujeres**.

---
### Ejercicio 5: Pruebas $Z$ de Proporciones Clínicas


In [ ]:
# -------------------------------------------------------------
# 1. Prueba Z para Una Proporción: Tabaquismo vs Referencia 15%
# H0: p = 0.15 vs H1: p != 0.15
# -------------------------------------------------------------
n_total = len(df)
n_fumadores = (df["smoker"] == "yes").sum()
p_muestral = n_fumadores / n_total
p_referencia = 0.15

z_stat_1p, p_val_1p = proportions_ztest(count=n_fumadores, nobs=n_total, value=p_referencia)

# -------------------------------------------------------------
# 2. Prueba Z para Dos Proporciones: Fumadores Hombres vs Mujeres
# H0: p_hombres = p_mujeres vs H1: p_hombres != p_mujeres
# -------------------------------------------------------------
hombres = df[df["sex"] == "male"]
mujeres = df[df["sex"] == "female"]

count_h = (hombres["smoker"] == "yes").sum()
count_m = (mujeres["smoker"] == "yes").sum()
nobs_h, nobs_m = len(hombres), len(mujeres)

z_stat_2p, p_val_2p = proportions_ztest(count=[count_h, count_m], nobs=[nobs_h, nobs_m])

print("=" * 65)
print("         PRUEBAS Z DE HIPÓTESIS PARA PROPORCIONES")
print("=" * 65)
print("1. PROPORCIÓN GENERAL DE FUMADORES VS 15% REFERENCIA:")
print(f"   Prevalencia Muestral: {p_muestral*100:.2f}% ({n_fumadores} de {n_total})")
print(f"   Estadístico Z:        {z_stat_1p:.3f} | p-valor: {p_val_1p:.4e}")
print(f"   Decisión:             {'Rechazar H0 (Mayor al 15%)' if p_val_1p < 0.05 else 'No rechazar H0'}")
print("-" * 65)
print("2. COMPARACIÓN DE PROPORCIÓN: HOMBRES VS MUJERES:")
print(f"   Fumadores Hombres:    {count_h/nobs_h*100:.2f}% ({count_h} de {nobs_h})")
print(f"   Fumadoras Mujeres:    {count_m/nobs_m*100:.2f}% ({count_m} de {nobs_m})")
print(f"   Estadístico Z:        {z_stat_2p:.3f} | p-valor: {p_val_2p:.4f}")
print(f"   Decisión:             {'Rechazar H0 (Diferencia significativa)' if p_val_2p < 0.05 else 'No hay diferencia significativa por género'}")
print("=" * 65)


## 7. Tema 4.4.3: Pruebas de Independencia ($\chi^2$ de Pearson)

### Marco Teórico:
La prueba de **Chi-cuadrada de Independencia** evalúa si dos variables categóricas están asociadas o son estadísticamente independientes:
- **$H_0$**: Las variables son independientes (no hay relación entre ellas).
- **$H_1$**: Las variables están asociadas (dependientes).

Fórmula del estadístico:
$$\chi^2 = \sum \frac{(O_{ij} - E_{ij})^2}{E_{ij}}$$
donde $O$ son las frecuencias observadas y $E$ son las esperadas bajo independencia ($E_{ij} = \frac{\text{Total Fila } i \times \text{Total Col } j}{N}$).

---
### Ejercicio 6: ¿Es el Hábito de Fumar Independiente de la Región Geográfica?


In [ ]:
# Creamos la tabla de contingencia entre smoker y region
tabla_contingencia = pd.crosstab(df["smoker"], df["region"])

chi2_stat, p_val_chi2, dof, esperados = stats.chi2_contingency(tabla_contingencia)
df_esperados = pd.DataFrame(esperados, index=tabla_contingencia.index, columns=tabla_contingencia.columns)

print("=" * 65)
print("     PRUEBA DE CHI-CUADRADA DE INDEPENDENCIA (smoker vs region)")
print("=" * 65)
print("Frecuencias Observadas:")
print(tabla_contingencia)
print()
print(f"Estadístico Chi2:      {chi2_stat:.3f}")
print("Grados de Libertad:   ", dof)
print(f"Valor p:               {p_val_chi2:.4f}")
print(f"Decisión (alpha=0.05): {'RECHAZAR H0 (Asociadas)' if p_val_chi2 < 0.05 else 'NO RECHAZAR H0 (Son independientes)'}")
print("=" * 65)

# Visualización con Heatmap de Frecuencias Observadas vs Esperadas
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.heatmap(tabla_contingencia, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
axes[0].set_title("Frecuencias Observadas (O)", fontweight="bold")

sns.heatmap(df_esperados, annot=True, fmt=".1f", cmap="Greens", cbar=False, ax=axes[1])
axes[1].set_title("Frecuencias Esperadas bajo H0 (E)", fontweight="bold")

plt.tight_layout()
plt.show()


## 8. Tema 4.4.4: Pruebas de Bondad de Ajuste y Normalidad

### El Supuesto de Normalidad:
Las pruebas paramétricas tradicionales ($t$ de Student, ANOVA) asumen que los datos provienen de una distribución normal.
- **Prueba de Shapiro-Wilk** (`stats.shapiro`) y **D'Agostino-Pearson** (`stats.normaltest`):
  - $H_0$: Los datos provienen de una distribución normal.
  - $H_1$: Los datos **no** provienen de una distribución normal.

### ¿Qué hacer si no hay normalidad?
Cuando la distribución es marcadamente asimétrica (como ocurre con los costos médicos en seguros), recurrimos a **pruebas no paramétricas** basadas en rangos:
- **Prueba de Mann-Whitney $U$**: La alternativa no paramétrica robusta a la prueba $t$ de dos muestras independientes.

---
### Ejercicio 7: Diagnóstico de Normalidad y Contraste Robusto Mann-Whitney $U$


In [ ]:
# 1. Pruebas de normalidad sobre 'charges' y 'bmi'
stat_dag_ch, p_dag_ch = stats.normaltest(df["charges"])
stat_dag_bmi, p_dag_bmi = stats.normaltest(df["bmi"])

# 2. Alternativa no paramétrica: Mann-Whitney U para fumadores vs no fumadores
u_stat, p_val_mwu = stats.mannwhitneyu(fumadores, no_fumadores, alternative="two-sided")

print("=" * 65)
print("     PRUEBAS DE NORMALIDAD Y CONTRASTE NO PARAMÉTRICO")
print("=" * 65)
print(f"Normalidad en 'charges' (p-valor D'Agostino): {p_dag_ch:.4e} -> ¡No es normal!")
print(f"Normalidad en 'bmi'     (p-valor D'Agostino): {p_dag_bmi:.4f} -> Distribución casi normal")
print("-" * 65)
print("CONTRASTE ROBUSTO DE MANN-WHITNEY U (Fumadores vs No Fumadores):")
print(f"Estadístico U: {u_stat:,.0f} | p-valor: {p_val_mwu:.4e}")
print(f"Conclusión:    {'Rechazar H0 (La diferencia de rangos es contundente)' if p_val_mwu < 0.05 else 'No rechazar H0'}")
print("=" * 65)

# Visualización de la asimetría de charges vs ajuste normal
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.histplot(df["charges"], kde=True, color="#34495e", bins=30, ax=axes[0])
axes[0].set_title("Histograma de Costos (Sesgo Positivo)", fontweight="bold")
axes[0].set_xlabel("Cargos Médicos (USD)")

stats.probplot(df["charges"], dist="norm", plot=axes[1])
axes[1].set_title("Gráfico Q-Q Normal (Desviación Severa)", fontweight="bold")
axes[1].get_lines()[0].set_color("#2980b9")
axes[1].get_lines()[1].set_color("#e74c3c")

plt.tight_layout()
plt.show()


## 9. Tema 4.8: Aplicaciones Prácticas y Dashboard Actuarial Multivariado

Integramos las decisiones estadísticas en un panel de control analítico para la toma de decisiones basada en datos.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# Panel 1: Violín de Costos según Hábito de Fumar (Prueba Welch + Mann Whitney)
sns.violinplot(data=df, x="smoker", y="charges", palette=["#2ecc71", "#e74c3c"], hue="smoker", legend=False, inner="quartile", ax=axes[0])
axes[0].set_title("1. Comparativa de Costos (p < 0.001***)", fontweight="bold")
axes[0].set_xlabel("Fumador")
axes[0].set_ylabel("Cargos (USD)")

# Panel 2: Obesidad + Tabaquismo (Interacción Crítica)
df["obesidad"] = np.where(df["bmi"] >= 30, "Obeso (BMI >= 30)", "No Obeso")
sns.barplot(data=df, x="obesidad", y="charges", hue="smoker", palette=["#27ae60", "#c0392b"], capsize=0.1, ax=axes[1])
axes[1].set_title("2. Interacción Obesidad x Tabaquismo", fontweight="bold")
axes[1].set_xlabel("Estado Ponderal")
axes[1].set_ylabel("Costo Promedio (USD)")

# Panel 3: Curva de Potencia de Decisión
ns = np.linspace(10, 200, 50)
pows = [analizador_potencia.power(effect_size=0.5, nobs1=n, alpha=0.05) for n in ns]
axes[2].plot(ns, pows, color="#8e44ad", lw=2.5)
axes[2].axhline(0.80, color="red", linestyle="--", label="Objetivo 80%")
axes[2].set_title("3. Curva de Potencia (Efecto d=0.5)", fontweight="bold")
axes[2].set_xlabel("Tamaño de Muestra (n)")
axes[2].set_ylabel("Potencia (1 - beta)")
axes[2].legend()

plt.suptitle("Dashboard Actuarial de Inferencia y Pruebas de Hipótesis", fontsize=14, fontweight="bold")
plt.subplots_adjust(top=0.88, wspace=0.3)
plt.show()


## 10. Guía Didáctica y Orientaciones Pedagógicas para el Docente

Esta sección final está diseñada para orientar tu exposición en el aula y fomentar el pensamiento crítico de los estudiantes:

---

### Preguntas Socráticas para el Debate en Clase:
1. **Sobre la Interpretación del Valor $p$ (Tema 4.1)**:
   - *Pregunta a los alumnos*: "Si obtenemos un valor $p = 0.03$, ¿significa que hay un 97% de probabilidad de que la hipótesis alternativa sea verdadera?"
   - *Orientación Docente*: **No**. El valor $p$ **no** es la probabilidad de que ninguna hipótesis sea cierta. Es la probabilidad de observar datos tan extremos bajo el supuesto de que $H_0$ es cierta. Asignar probabilidad directa a una hipótesis requiere inferencia bayesiana.
2. **Sobre Significancia Estadística vs Relevancia Práctica (Tema 4.3)**:
   - *Pregunta*: "Si analizamos 2 millones de pólizas y encontramos que los clientes con ojos claros pagan $12,001 USD frente a $12,000 USD de los de ojos oscuros con un $p < 0.00001$, ¿debemos cambiar las tarifas del seguro?"
   - *Orientación Docente*: No. Con muestras masivas ($n \to \infty$), cualquier diferencia microscópica resulta estadísticamente significativa ($p < \alpha$), pero su tamaño del efecto ($d$ de Cohen) es nulo ($d \approx 0.0001$). Siempre debemos acompañar el valor $p$ con el tamaño del efecto y el criterio de negocio.
3. **Sobre el Peligro del *p-Hacking* y Pruebas Múltiples (Tema 4.4)**:
   - *Pregunta*: "¿Qué ocurre si realizamos 100 pruebas de hipótesis con $\alpha = 0.05$ buscando diferencias en cientos de variables al azar?"
   - *Orientación Docente*: Por puro azar esperamos obtener aproximadamente $100 \times 0.05 = 5$ resultados falsamente significativos (Error Tipo I). Explica la necesidad de correcciones (como la corrección de Bonferroni o FDR de Benjamini-Hochberg) cuando se hacen comparaciones múltiples.
4. **Sobre la Elección entre Pruebas Paramétricas y No Paramétricas (Tema 4.4.4)**:
   - *Pregunta*: "¿Por qué realizamos la prueba de Mann-Whitney $U$ después de haber hecho la prueba $t$ de Welch en los costos médicos?"
   - *Orientación Docente*: Los costos médicos presentan una asimetría positiva severa y colas pesadas (rechazo de normalidad con $p < 10^{-10}$). Aplicar Mann-Whitney $U$ demuestra a los alumnos cómo validar que las conclusiones no son artefactos de la violación de supuestos teóricos.

---
### Guía Rápida para Selección de Pruebas Estadísticas:
| Pregunta de Investigación | Tipo de Variable | Prueba Estadística en Python |
| :--- | :--- | :--- |
| ¿Una media difiere de un estándar $\mu_0$? | Cuantitativa continua | `stats.ttest_1samp` |
| ¿Dos grupos independientes tienen medias distintas? | Cuantitativa vs Categórica (2 niveles) | `stats.ttest_ind(..., equal_var=False)` (Welch) |
| ¿Tres o más grupos tienen medias distintas? | Cuantitativa vs Categórica ($k \ge 3$) | `stats.f_oneway` (ANOVA) |
| ¿Dos grupos difieren sin asumir normalidad? | Cuantitativa ordinal o no normal | `stats.mannwhitneyu` |
| ¿Una tasa difiere de una referencia $p_0$? | Binaria / Proporción | `proportions_ztest` |
| ¿Dos variables categóricas están asociadas? | Categórica vs Categórica | `stats.chi2_contingency` |

---
### Conexión con la Unidad 5 (Modelado y Regresión):
Explica a tus alumnos que la prueba $t$ de Welch y el ANOVA son en realidad casos particulares del **Modelo Lineal General (Regresión Lineal)**:
$$\text{charges} = \beta_0 + \beta_1 (\text{smoker}) + \epsilon$$
Donde probar si $\beta_1 = 0$ mediante una prueba $t$ sobre el coeficiente de regresión equivale exactamente a la prueba de hipótesis entre grupos que realizamos en esta unidad.
